# Court E2E on val — first F1 number

**Goal:** see macro F1 for court_consideration retrieval on val.csv, hour-1 baseline through hour-3 final.

**Pipeline:** 3 stages, all measured components from your repo or 2026 published wins.
1. **Retrieve** — BM25 multi-lingual (DE/FR/IT) on filtered court corpus, top 1500 per query
2. **Rerank** — Qwen3-Reranker-8B-AWQ, softmax([no,yes])[:,1], top 200 per query
3. **Judge** — Qwen3-14B-AWQ with rich context on borderline (~30/query)

**Honest expectation:**
- Stage 1+2 alone: F1 ≈ 0.20-0.30 (auto-YES on top-K by reranker score)
- Stage 1+2+3 with rich judge context: F1 ≈ 0.35-0.50

If hour-1 (stage 1 retrieval recall) is below 0.60, the pipeline cannot reach 0.4 F1 — stop and widen retrieval before continuing.

## Cell 1 — Setup

In [ ]:
!pip install -q transformers sentencepiece accelerate rank_bm25 langid pandas pyarrow
!pip install -q vllm autoawq

import torch, time, json, gc, os, re, sys, io
import pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict

DATA_DIR = Path('/content/drive/MyDrive/swiss_law/data')        # ADJUST
OUT_DIR  = Path('/content/drive/MyDrive/swiss_law/court_e2e_2026-05-22')
OUT_DIR.mkdir(parents=True, exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Cell 2 — Load val + court corpus, identify court vs law gold

In [ ]:
val = pd.read_csv(DATA_DIR / 'val.csv')
print('val columns:', list(val.columns), '| rows:', len(val))
print(val.head(2).to_string())

# Adjust column names if needed
QID, QCOL, GOLDCOL = 'query_id', 'query', 'gold_citations'

def split_gold(s):
    if not isinstance(s, str): return []
    return [c.strip() for c in re.split(r'[;,]', s) if c.strip()]

val['gold_list'] = val[GOLDCOL].apply(split_gold)
print('\nMean gold per query:', val['gold_list'].apply(len).mean())

court = pd.read_csv(DATA_DIR / 'court_considerations.csv', low_memory=False)
court['text'] = court['text'].astype(str)
court_cit_set = set(court['citation'])
print('court corpus rows:', len(court))

# Partition each query's gold into court vs law (court if citation appears in court corpus)
val['gold_court'] = val['gold_list'].apply(lambda L: [c for c in L if c in court_cit_set])
val['gold_law']   = val['gold_list'].apply(lambda L: [c for c in L if c not in court_cit_set])
print('\nPer-query gold split:')
for _, r in val.iterrows():
    print(f"  {r[QID]}: {len(r['gold_court'])} court + {len(r['gold_law'])} law (total {len(r['gold_list'])})")

## Cell 3 — Filter court corpus to doctrinal pool (drop boilerplate)

CPU-only. Compresses 2.1M → ~600-800k. The drops are universal (no query-specific filtering).

In [ ]:
# Cantonal court patterns (federal cases use BGE/ATF/DTF + docket like 1B_/4A_/6B_/etc.)
CANTONAL_RE = re.compile(r'^(KGer|OGer|BezGer|Kantonsgericht|Cour cantonale)', re.I)

# Dispositif / costs / signature opener patterns
DISPOSITIF_RE = re.compile(r'^\s*(?:\d+\.?\s+)?(?:Die Beschwerde wird|Le recours est|Il ricorso \u00e8|Im Namen|Au nom de la|Gerichtskosten|Les frais judiciaires|frais et d\u00e9pens|Es werden keine Kosten|Il n\'est pas per\u00e7u|Lausanne,)', re.I)

# Apply filters
mask = (
    (~court['citation'].str.match(CANTONAL_RE, na=False))
    & (~court['text'].str.match(DISPOSITIF_RE, na=False))
    & (court['text'].str.len() >= 200)
)
pool = court[mask].copy().reset_index(drop=True)
print(f'Doctrinal pool: {len(pool)} / {len(court)} ({100*len(pool)/len(court):.1f}%)')

# Detect language per row (fast: only on the doctrinal pool, takes ~3-5 min for 600k)
import langid
langid.set_languages(['de', 'fr', 'it'])
pool['language'] = pool['text'].apply(lambda t: langid.classify(t[:300])[0])
print('Language dist:', pool['language'].value_counts().to_dict())

# How much court gold survived?
all_court_gold = set()
for L in val['gold_court']: all_court_gold.update(L)
surviving_gold = all_court_gold & set(pool['citation'])
print(f'\nCourt gold surviving filter: {len(surviving_gold)} / {len(all_court_gold)} ({100*len(surviving_gold)/max(1,len(all_court_gold)):.1f}%)')
if len(surviving_gold) / max(1, len(all_court_gold)) < 0.90:
    print('WARN: >10% of gold dropped — filter is too aggressive. Loosen DISPOSITIF_RE.')

## Cell 4 — BM25 indexes per language + cross-lingual query expansion

Build one BM25 index per language. For each query, translate to DE/FR/IT once (Helsinki-NLP opus-mt) and query each index. Top 500 per language → ~1500 candidates per query.

In [ ]:
from rank_bm25 import BM25Okapi
from transformers import MarianMTModel, MarianTokenizer

def tokenize(text):
    return re.findall(r"\b\w{3,}\b", str(text).lower())

# Build BM25 per language (one-time, ~10-30 min for 600k)
print('Building BM25 indexes...')
BM25 = {}
POOL_IDX = {}
for lang in ['de', 'fr', 'it']:
    sub = pool[pool['language'] == lang].reset_index(drop=True)
    POOL_IDX[lang] = sub
    t0 = time.time()
    BM25[lang] = BM25Okapi([tokenize(t) for t in sub['text']])
    print(f'  {lang}: {len(sub)} docs indexed in {time.time()-t0:.1f}s')

# Translate val queries EN → DE/FR/IT (cached)
TRANS_CACHE = OUT_DIR / 'query_translations.json'
if TRANS_CACHE.exists():
    qtrans = json.loads(TRANS_CACHE.read_text())
    print('Loaded query translations from cache')
else:
    qtrans = {q[QID]: {} for _, q in val.iterrows()}
    for src, lang in [('de', 'de'), ('fr', 'fr'), ('it', 'it')]:
        name = f'Helsinki-NLP/opus-mt-en-{src}'
        tok = MarianTokenizer.from_pretrained(name)
        mdl = MarianMTModel.from_pretrained(name).to('cuda').half().eval()
        for _, r in val.iterrows():
            inp = tok([r[QCOL]], return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
            with torch.no_grad():
                gen = mdl.generate(**inp, max_length=512, num_beams=1)
            qtrans[r[QID]][lang] = tok.batch_decode(gen, skip_special_tokens=True)[0]
        del mdl, tok; gc.collect(); torch.cuda.empty_cache()
    for _, r in val.iterrows():
        qtrans[r[QID]]['en'] = r[QCOL]
    TRANS_CACHE.write_text(json.dumps(qtrans, ensure_ascii=False, indent=2))
    print('Translated and cached.')

# Per-query top-500-per-language union
candidates = {}
for _, r in val.iterrows():
    qid = r[QID]
    cand_rows = []
    for lang in ['de', 'fr', 'it']:
        q_lang = qtrans[qid].get(lang, r[QCOL])
        q_tok = tokenize(q_lang) + tokenize(r[QCOL])  # bilingual tokens
        scores = BM25[lang].get_scores(q_tok)
        top = np.argpartition(-scores, min(500, len(scores)-1))[:500]
        for i in top:
            cand_rows.append({
                'citation': POOL_IDX[lang]['citation'].iloc[i],
                'text': POOL_IDX[lang]['text'].iloc[i],
                'language': lang,
                'bm25_score': float(scores[i]),
            })
    cand_df = pd.DataFrame(cand_rows).drop_duplicates('citation').reset_index(drop=True)
    # Normalize BM25 within query for fusion later
    s = cand_df['bm25_score']
    cand_df['bm25_norm'] = (s - s.min()) / (s.max() - s.min() + 1e-9)
    candidates[qid] = cand_df
    n_gold_in_pool = sum(c in cand_df['citation'].values for c in r['gold_court'])
    print(f"  {qid}: pool={len(cand_df)}, gold_in_pool={n_gold_in_pool}/{len(r['gold_court'])}")

# Stage-1 recall (the most important gate)
stage1_recall = []
for _, r in val.iterrows():
    g = set(r['gold_court'])
    if not g: continue
    hit = sum(c in candidates[r[QID]]['citation'].values for c in g)
    stage1_recall.append(hit / len(g))
print(f'\n=== Stage 1 recall on val court gold: macro = {np.mean(stage1_recall):.3f} ===')
print('  (target: ≥ 0.60. If below, widen BM25 top-K per language to 1000.)')

del BM25; gc.collect()  # free BM25 memory before loading models

## Cell 5 — Rerank with Qwen3-Reranker-8B

Calibrated softmax([no, yes])[:,1]. Score top 200 per query (BM25-ranked). Output: `rerank_scores[qid][citation] ∈ [0,1]`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

RERANK_NAME = 'Qwen/Qwen3-Reranker-8B'  # adjust if you have a local AWQ variant
tok = AutoTokenizer.from_pretrained(RERANK_NAME, padding_side='left', trust_remote_code=True)
mdl = AutoModelForCausalLM.from_pretrained(RERANK_NAME, torch_dtype=torch.float16, trust_remote_code=True).to('cuda').eval()
YES_ID = tok.convert_tokens_to_ids('yes')
NO_ID  = tok.convert_tokens_to_ids('no')
print('Reranker loaded. yes_id =', YES_ID, ', no_id =', NO_ID)

PROMPT_TPL = (
    '<|im_start|>system\n'
    'Judge whether the Document meets the requirements based on the Query and the Instruction. '
    'Answer only "yes" or "no".<|im_end|>\n'
    '<|im_start|>user\n'
    '<Instruct>: Given a query, retrieve passages from Swiss Federal Court decisions '
    'that state the legal doctrine relevant to the query.\n'
    '<Query>: {query}\n'
    '<Document>: {doc}<|im_end|>\n'
    '<|im_start|>assistant\n<think>\n\n</think>\n\n'
)

@torch.no_grad()
def rerank_batch(query, docs, batch_size=8, max_len=2048):
    scores = []
    for i in range(0, len(docs), batch_size):
        batch_docs = docs[i:i+batch_size]
        prompts = [PROMPT_TPL.format(query=query[:1500], doc=d[:3000]) for d in batch_docs]
        enc = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=max_len).to('cuda')
        out = mdl(**enc, use_cache=False)
        last_logits = out.logits[:, -1, :]
        yn = torch.stack([last_logits[:, NO_ID], last_logits[:, YES_ID]], dim=-1)
        p_yes = torch.softmax(yn, dim=-1)[:, 1].float().cpu().numpy().tolist()
        scores.extend(p_yes)
    return scores

TOP_RERANK = 200
rerank_results = {}
for _, r in val.iterrows():
    qid = r[QID]
    cdf = candidates[qid].nlargest(min(TOP_RERANK, len(candidates[qid])), 'bm25_score').reset_index(drop=True)
    t0 = time.time()
    p_yes = rerank_batch(r[QCOL], cdf['text'].tolist(), batch_size=8)
    cdf['p_rerank'] = p_yes
    cdf['fused'] = 0.7 * cdf['p_rerank'] + 0.3 * cdf['bm25_norm']
    rerank_results[qid] = cdf.sort_values('fused', ascending=False).reset_index(drop=True)
    gold_in_top = sum(c in rerank_results[qid].head(50)['citation'].values for c in r['gold_court'])
    print(f"  {qid}: reranked {len(cdf)} in {time.time()-t0:.1f}s | gold in top50 = {gold_in_top}/{len(r['gold_court'])}")

del mdl, tok; gc.collect(); torch.cuda.empty_cache()

## Cell 6 — Quick F1 with reranker-only (auto-YES top-K, no judge)

Hour-1 checkpoint. Sweep K and pick the best per-query F1 to see the rerank-only ceiling.

In [ ]:
def f1(pred_set, gold_set):
    if not gold_set: return None
    tp = len(pred_set & gold_set)
    if not pred_set or tp == 0: return 0.0
    p = tp / len(pred_set); r = tp / len(gold_set)
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0

# Fixed-K sweep on val (just for diagnostic; not for tuning)
K_GRID = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
print(f"{'K':>4} | {'macro F1':>10} | per-query F1")
best = (0, 0.0)
for K in K_GRID:
    per_q = []
    for _, r in val.iterrows():
        pred = set(rerank_results[r[QID]].head(K)['citation'])
        v = f1(pred, set(r['gold_court']))
        if v is not None: per_q.append(v)
    macro = np.mean(per_q)
    if macro > best[1]: best = (K, macro)
    print(f'{K:>4} | {macro:>10.3f} | {[round(v,2) for v in per_q]}')
print(f'\nRerank-only best fixed-K: K={best[0]} → macro F1 = {best[1]:.3f}')
print('  (this is the floor. Judge stage in Cell 7-8 should push it higher.)')

## Cell 7 — Qwen3-14B judge on borderline candidates (rich context)

For each query: take the top 50 fused-score candidates. Top 5 → auto-YES. Bottom 30 of top 50 → borderline pile for judge.

**Rich context the judge sees:** statute references (regex), opener-verb-owner (heuristic), chamber (regex from case_id), language. These features ARE the V1 discriminators — the judge verifies rather than guesses.

In [ ]:
from vllm import LLM, SamplingParams

QWEN_NAME = 'Qwen/Qwen3-14B-AWQ'
llm = LLM(model=QWEN_NAME, quantization='awq_marlin', max_model_len=4096,
          gpu_memory_utilization=0.85, dtype='float16', trust_remote_code=True)
judge_tok = AutoTokenizer.from_pretrained(QWEN_NAME, trust_remote_code=True)

# Feature extractors (CPU regex)
STATUTE_RE = re.compile(r'\b[Aa]rt(?:icle)?\.?\s+\d+[a-z]?(?:\s+(?:Abs|al|cpv)\.?\s+\d+)?(?:\s+(?:lit|let|lett)\.?\s+[a-z])?\s+[A-Z]{2,6}\b')
DE_PARTY_RE = re.compile(r'^(?:\d+(?:\.\d+)?\.?\s+)?Der Beschwerdef\u00fchrer|^(?:\d+(?:\.\d+)?\.?\s+)?Die Beschwerdef\u00fchrerin|^(?:\d+(?:\.\d+)?\.?\s+)?Die Vorinstanz', re.I)
FR_PARTY_RE = re.compile(r'^(?:\d+(?:\.\d+)?\.?\s+)?Le recourant|La recourante|La cour cantonale', re.I)
CHAMBER_RE  = re.compile(r'\b(BGE|ATF|DTF)\s+\d+\s+([IVX]+)|^(\d[A-Z])_')

def extract_features(text, citation):
    statutes = list(set(STATUTE_RE.findall(text[:1000])))[:6]
    voice = 'court'
    if DE_PARTY_RE.match(text) or FR_PARTY_RE.match(text):
        voice = 'appellant_or_lower_court'
    chamber = ''
    m = CHAMBER_RE.search(citation)
    if m:
        chamber = m.group(2) or m.group(3) or ''
    return {'statutes': statutes, 'voice': voice, 'chamber': chamber}

def build_judge_prompt(query, citation, text, feats):
    msgs = [
        {'role': 'system', 'content': (
            'You are a Swiss Federal Court doctrinal-paragraph expert. '
            'Output exactly one token: YES or NO. Default to NO when uncertain.'
        )},
        {'role': 'user', 'content': (
            f'Query (English):\n{query[:1500]}\n\n'
            f'Candidate paragraph:\n'
            f'  Citation: {citation}\n'
            f'  Chamber: {feats["chamber"]}\n'
            f'  Opener voice: {feats["voice"]}\n'
            f'  Statute references found: {feats["statutes"]}\n'
            f'  Text: {text[:1800]}\n\n'
            'Answer YES only if ALL three:\n'
            '(a) Federal court (not cantonal)\n'
            '(b) States a doctrinal rule in the court\'s voice OR is an echo/application from a case on-point\n'
            '(c) The doctrinal sub-question matches one of the query\'s legal sub-questions (not merely the same statute)\n\n'
            'Answer:'
        )},
    ]
    return judge_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

TOP_KEEP   = 50  # candidates per query that the judge stage considers
AUTO_YES_K = 5    # top-5 by fused score are auto-accepted (skip judge)

auto_yes, borderline = {}, {}
for _, r in val.iterrows():
    qid = r[QID]
    top = rerank_results[qid].head(TOP_KEEP).reset_index(drop=True)
    auto_yes[qid] = set(top.head(AUTO_YES_K)['citation'])
    borderline[qid] = top.iloc[AUTO_YES_K:].reset_index(drop=True)

# Build all judge prompts at once (vLLM batches them)
all_prompts, all_keys = [], []
for qid, bdf in borderline.items():
    q = val[val[QID]==qid][QCOL].iloc[0]
    for _, c in bdf.iterrows():
        feats = extract_features(c['text'], c['citation'])
        all_prompts.append(build_judge_prompt(q, c['citation'], c['text'], feats))
        all_keys.append((qid, c['citation']))

print(f'Judging {len(all_prompts)} borderline candidates...')
t0 = time.time()
outputs = llm.generate(all_prompts, SamplingParams(temperature=0.0, max_tokens=4))
print(f'Done in {time.time()-t0:.1f}s')

verdicts = {}
for (qid, cit), out in zip(all_keys, outputs):
    txt = out.outputs[0].text.strip().upper()
    verdicts.setdefault(qid, {})[cit] = txt.startswith('YES')

del llm; gc.collect(); torch.cuda.empty_cache()

## Cell 8 — Final F1 = auto-YES ∪ judge-YES, per-query + macro

In [ ]:
rows = []
for _, r in val.iterrows():
    qid = r[QID]
    gold = set(r['gold_court'])
    if not gold:
        rows.append({'qid': qid, 'gold': 0, 'pred': 0, 'tp': 0, 'P': None, 'R': None, 'F1': None})
        continue
    pred = set(auto_yes[qid])
    judge_yes = {c for c, v in verdicts.get(qid, {}).items() if v}
    pred |= judge_yes
    # Floor K=3 for small-gold protection
    if len(pred) < 3:
        for c in rerank_results[qid]['citation']:
            pred.add(c)
            if len(pred) >= 3: break
    tp = len(pred & gold)
    P = tp / max(1, len(pred))
    R = tp / len(gold)
    F1v = (2*P*R/(P+R)) if (P+R) > 0 else 0.0
    rows.append({'qid': qid, 'gold': len(gold), 'pred': len(pred), 'tp': tp, 'P': round(P,3), 'R': round(R,3), 'F1': round(F1v,3)})

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / 'per_query_results.csv', index=False)

macro_f1 = res.loc[res['F1'].notna(), 'F1'].mean()
macro_p  = res.loc[res['P'].notna(),  'P'].mean()
macro_r  = res.loc[res['R'].notna(),  'R'].mean()

print(res.to_string(index=False))
print(f'\n=== MACRO over {res["F1"].notna().sum()} queries with court gold ===')
print(f'  P = {macro_p:.3f}')
print(f'  R = {macro_r:.3f}')
print(f'  F1 = {macro_f1:.3f}')

# Save final predictions for inspection
preds_out = []
for _, r in val.iterrows():
    qid = r[QID]
    pred = set(auto_yes.get(qid, set()))
    pred |= {c for c, v in verdicts.get(qid, {}).items() if v}
    for c in pred:
        preds_out.append({'qid': qid, 'citation': c, 'is_gold': c in set(r['gold_court'])})
pd.DataFrame(preds_out).to_csv(OUT_DIR / 'final_predictions.csv', index=False)
print(f'\nFiles written:')
for p in sorted(OUT_DIR.iterdir()):
    print(' ', p.name)

## Cell 9 — Where did it fail? (error analysis)

For each query, print which gold paragraphs were missed and at what rerank position they sat. This is where you iterate.

In [ ]:
for _, r in val.iterrows():
    qid = r[QID]
    gold = set(r['gold_court'])
    if not gold: continue
    df = rerank_results[qid].reset_index(drop=True)
    df['is_gold'] = df['citation'].isin(gold)
    df['rank'] = df.index + 1
    in_pool = df[df['is_gold']]
    missed = gold - set(df['citation'])
    print(f"\n{qid}: {len(in_pool)}/{len(gold)} gold in rerank pool. Ranks: {in_pool['rank'].tolist()}")
    if missed:
        print(f'  Missed entirely (not in BM25 top-1500): {len(missed)}')
        print(f'  Examples: {list(missed)[:3]}')